Below is the code implementing the Full Primal, Ex Post Allocation, DSIC Linear Program. Just run the following cell to initialize the code.

In [4]:
from gurobipy import Model, GRB, quicksum
import numpy as np


def Primal(V, f):

    # Step 1: Create a new model
    model = Model("maximize_function")

    # Example dimensions for variables and vectors
    num_i = len(V[0])  # Number of bidders i
    num_j = len(V[0][0])  # Number of items j

    # Step 2: Define decision variables for y and p
    y = model.addVars(len(V), num_i, num_j, vtype=GRB.CONTINUOUS, name="y")  # y_i(v) as a decision variable
    p = model.addVars(len(V), num_i, vtype=GRB.CONTINUOUS, name="p")  # p_i(v) as a decision variable

    # Step 3: Define the function f(v)
    # Step 4: Set the objective
    model.setObjective(
        quicksum(
        f[v] * quicksum(quicksum(y[v, i, j] * V[v][i][j] for j in range(num_j)) - p[v, i] for i in range(num_i))
        for v in range(len(V))
    ), 
    GRB.MAXIMIZE
    )

    # Step 5: Add constraints
    # Example: constraints on y and p (adjust based on your problem)
    
    model.addConstrs((y[v,i,j] == y[v,i,j+1] 
                      for i in range(num_i-1) 
                      for j in range(num_j-1) 
                      for v in range(len(V))
                      if V[v][i][j]-V[v][i+1][j] == V[v][i][j+1]- V[v][i+1][j+1])
        
    ,"Symmetry")
    

    model.addConstrs(
      ((quicksum(y[v, i, j] * V[v][i][j] for j in range(num_j)) - p[v,i] >= quicksum(y[v_p, i, j] * V[v][i][j] for j in range(num_j))- p[v_p, i])
       for v_p in range(len(V)) 
       for v in range(len(V)) 
       for not_i in range(num_i)
       for i in range(num_i)
       if not_i != i
       if np.all(V[v][not_i]==V[v_p][not_i])
       
        ), "DSIC")

    model.addConstrs((quicksum(y[v, i, j] * V[v][i][j] for j in range(num_j)) - p[v, i] >= 0
                      for v in range(len(V))
                      for i in range(num_i)
                 ), "IR")


    model.addConstrs((quicksum(y[v, i, j] for i in range(num_i)) <= 1
                      for v in range(len(V))
                      for j in range(num_j)
                 ), "item feasibility")


    model.addConstrs((quicksum(y[v, i, j] for j in range(num_j)) <= 1
                      for v in range(len(V))
                      for i in range(num_i)
                 ), "UD bidder feasibility")


    model.addConstrs((y[v, i, j] >= 0
                      for v in range(len(V))
                      for j in range(num_j)
                      for i in range(num_i)
                 ), "nonneg_y")

    model.addConstrs((p[v, i] >= 0
                      for v in range(len(V))
                      for i in range(num_i)
                 ), "nonneg_p")

    # Step 6: Optimize the model
    model.optimize()

    # Step 7: Print results
    if model.status == GRB.OPTIMAL:
        print(f"Optimal objective value: {model.objVal}")
        for v in range(len(V)):
            for i in range(num_i):
                for j in range(num_j):
                    print(f"y[{v},{i},{j}] = {y[v, i, j].X}, p[{v},{i}] = {p[v, i].X}")
    else:
        return("No optimal solution found.")
    return

Below are the parameters that we will enter into our linear program. They are currently set to have the type space be uniform [1,2] for a two bidder two item scenario. V contains the valuations for the bidders. f is the distribution of each v in V. f should sum to 1. It currently represents a uniform distribution. To solve the linear program with these parameters, just run the cell below. Reading the output of the program, the Optimal objective value will display the optimal output from the parameters. The display will then show each valuation and the allocation they receive. For example, y[3,0,1] = 1.0 means that in the fourth v (we start counting from 0), the first bidder (bidder 0) has an allocation of 1.0 for the second item (item 1).

In [5]:
V=[[[1,2],[1,2]],
   [[1,2],[1,1]],
   [[1,2],[2,2]],
   [[1,2],[2,1]],
  [[1,1],[1,2]],
  [[1,1],[1,1]],
  [[1,1],[2,2]],
  [[1,1],[2,1]],
   [[2,2],[1,2]],
   [[2,2],[1,1]],
   [[2,2],[2,2]],
   [[2,2],[2,1]],
  [[2,1],[1,2]],
  [[2,1],[1,1]],
  [[2,1],[2,2]],
  [[2,1],[2,1]]]  # v matrices
f = [1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16]  #distribution for which v is being chosen

Primal(V,f)

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[x86] - Darwin 23.6.0 23G93)

CPU model: Intel(R) Core(TM) i5-1038NG7 CPU @ 2.00GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 326 rows, 96 columns and 908 nonzeros
Model fingerprint: 0x4a6c5e3f
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [6e-02, 1e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 140 rows and 6 columns
Presolve time: 0.01s
Presolved: 186 rows, 90 columns, 746 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.1875000e+00   4.850000e+01   0.000000e+00      0s
      23    3.3750000e+00   0.000000e+00   0.000000e+00      0s

Solved in 23 iterations and 0.02 seconds (0.00 work units)
Optimal objective  3.375000000e+00
Optimal objective value: 3.375
y[0,0,0] = 0.5, p[0,0] = 0.0
y[0,0,1] = 0.5, p[0,0] = 0.0
y[0,1,0] = 0.5, p[0,1] = 0.0
y[0,1,1] = 0.5, p[0,1

Below is a welfare calculator. It takes in the V from the cell above and calculates what welfare should be with only item and bidder feasibility constrainst. To run the code, just run the cell below, and if you want to change the valuations, you can change the V in the cell above and run both cells again.

In [6]:
import itertools

def max_non_column_sum(matrices):
    total_max_sum = 0
    
    for matrix in matrices:
        row1 = matrix[0]
        row2 = matrix[1]
        
        # Find the maximum possible sum with non-overlapping columns
        max_sum = 0
        
        for col1 in range(len(row1)):
            for col2 in range(len(row2)):
                if col1 != col2:  # Ensure different columns
                    current_sum = row1[col1] + row2[col2]
                    max_sum = max(max_sum, current_sum)
        
        total_max_sum += max_sum  # Add the maximum sum of this matrix to the total
        
    return total_max_sum

# Generate matrices and find the total maximum sum
result = max_non_column_sum(V)
print("Total maximum non-column sum:", result)
print("Optimal value:", result/len(V))


Total maximum non-column sum: 54
Optimal value: 3.375
